# NB01 · 数据集解剖（mini-profiler）

| | |
|---|---|
| **目标** | 不看文档，用代码逼一个真实机器人数据集交代自己的一切——这是 Robot Dataset Profiler 的需求来源 |
| **前置** | NB00 完成 |
| **预计耗时** | 半天 |
| **产出物** | `results/NB01.json`（数据集档案）+ 3 张图 |
| **通过标准** | 能回答文末五个审问问题，且 profile 落盘 |

规则：从上到下顺序执行；每个 ✅ 检查点必须核对；最后的复盘必须填写并 commit。


In [ ]:
REPO_ID = "lerobot/pusht"   # 之后换 aloha / droid 子集重跑本笔记本，就是 Profiler 的泛化测试

import numpy as np
import matplotlib.pyplot as plt
import nbutils

ds = nbutils.load_lerobot_dataset(REPO_ID)
print(f"episodes={ds.num_episodes}  frames={ds.num_frames}  fps={ds.fps}")
print("features:", list(ds.features.keys()))

In [ ]:
# 第一帧：每个字段的形状、类型、取值范围——先知道数据长什么样
frame = ds[0]
for k, v in frame.items():
    if hasattr(v, "shape"):
        vmin = float(v.min()) if v.numel() else None
        vmax = float(v.max()) if v.numel() else None
        print(f"{k:30s} shape={tuple(v.shape)!s:18s} dtype={v.dtype}  range=[{vmin:.3g}, {vmax:.3g}]")
    else:
        print(f"{k:30s} {type(v).__name__} = {v}")

In [ ]:
# episode 长度分布——长度差异本身就是信息（任务难度不均？采集员风格？截断策略？）
try:
    ep_meta = ds.meta.episodes            # 新版 API
    lengths = np.array([e["length"] for e in ep_meta.values()]) if isinstance(ep_meta, dict) else np.array([e["length"] for e in ep_meta])
except Exception:
    # 兜底：从 frame 的 episode_index 数出来（慢但永远可用）
    idx = np.array([ds[i]["episode_index"].item() for i in range(0, ds.num_frames, max(1, ds.num_frames // 5000))])
    lengths = np.bincount(idx) * max(1, ds.num_frames // 5000)

plt.figure(figsize=(7, 3))
plt.hist(lengths, bins=30)
plt.xlabel("episode length (frames)"); plt.ylabel("count"); plt.title(f"{REPO_ID}: episode length distribution")
plt.savefig("results/NB01_lengths.png", dpi=120, bbox_inches="tight"); plt.show()
print(f"len: min={lengths.min()} median={np.median(lengths):.0f} max={lengths.max()}")

In [ ]:
# action 逐维统计 + 一条轨迹可视化：action 到底是什么物理量？
ep0 = [ds[i] for i in range(int(lengths[0]))]
actions = np.stack([f["action"].numpy() for f in ep0])
states  = np.stack([f["observation.state"].numpy() for f in ep0])
print("action dims:", actions.shape[1])
for d in range(actions.shape[1]):
    print(f"  dim{d}: mean={actions[:,d].mean():8.3f}  std={actions[:,d].std():7.3f}  range=[{actions[:,d].min():.2f}, {actions[:,d].max():.2f}]")

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(actions); ax[0].set_title("episode 0: action trajectory"); ax[0].set_xlabel("t")
ax[1].plot(states);  ax[1].set_title("episode 0: state trajectory");  ax[1].set_xlabel("t")
plt.savefig("results/NB01_traj.png", dpi=120, bbox_inches="tight"); plt.show()
# ✅ 检查点：action 曲线和 state 曲线相位差多少？谁领先谁？这告诉你 action 是"目标位置"还是"当前位置"。

In [ ]:
# 时间完整性：时间戳间隔是否严格等于 1/fps？缺帧多少？
ts = np.array([f["timestamp"].item() for f in ep0])
gaps = np.diff(ts)
expected = 1.0 / ds.fps
bad = np.abs(gaps - expected) > 0.2 * expected
print(f"expected gap={expected*1000:.1f}ms  actual: mean={gaps.mean()*1000:.1f}ms  异常间隔数={bad.sum()}/{len(gaps)}")
# ✅ 检查点：若异常间隔 > 0，这些位置的 (obs, action) 对还成立吗？训练时会发生什么？

In [ ]:
# success / reward 语义：成功是谁定义的、怎么写进数据的？
reward_keys = [k for k in frame.keys() if "reward" in k or "success" in k or "done" in k]
print("reward-ish keys:", reward_keys)
last = ep0[-1]
for k in reward_keys:
    print(f"  episode 0 最后一帧 {k} = {last[k]}")
# pusht 的 success 定义在环境里（coverage 阈值），数据集只存 reward——
# ✅ 检查点：这意味着"数据集里的 demo 全是成功的吗"这个问题，你要怎么回答？用代码验证。

In [ ]:
# 汇总落盘：这就是 Profiler v0 输出格式的雏形
profile = {
    "repo_id": REPO_ID,
    "episodes": int(ds.num_episodes), "frames": int(ds.num_frames), "fps": float(ds.fps),
    "features": list(ds.features.keys()),
    "ep_len": {"min": int(lengths.min()), "median": float(np.median(lengths)), "max": int(lengths.max())},
    "action_dim": int(actions.shape[1]),
    "timestamp_anomalies_ep0": int(bad.sum()),
}
nbutils.log_result("NB01", profile)

## 分析：五个审问问题（逐一书面回答）

1. **action 的物理语义**：目标位置还是位置增量？什么坐标系？单位？你的证据是哪张图/哪个数字？
2. **success 是谁定义的**：数据集、环境、还是评测脚本？改了阈值数字会怎样？
3. **fps 与控制频率**：录制 10fps 意味着策略以 10Hz 决策吗？真机上呢？
4. **缺帧的下游代价**：一个异常时间间隔对 BC 训练的 (o_t, a_t) 配对意味着什么？
5. **这个数据集缺什么**：如果你要用它训练一个鲁棒的 policy，最想补的 20 条 episode 是什么样的？

> 回答放在下方或 `week01/NOTES.md`。这五问换一个数据集（aloha、droid）再问一遍，答案的差异就是你 Profiler 的功能列表。


## 复盘（必填，不填不算完成这本 notebook）

> 复盘写在这里并 commit。允许粗糙，禁止事后美化。

- **预期 vs 实际**：
- **最大的一个意外**：
- **卡最久的一步和根因**：
- **用一句话向非技术人解释本次学到的东西**：
- **进入下一本之前要做的一个动作**：
